# Fragile Region Analysis

In [ ]:
import re
import pandas as pd

In [ ]:
WINDOW_SIZE = 100
STEP_SIZE = 20

def parse_sequence(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    seq = "".join(line for line in lines if not line.startswith(">"))
    seq = re.sub(r"[^ACGTacgt]", "", seq).upper()
    return seq

def gc_content(seq):
    if not seq:
        return 0.0
    return ((seq.count("G") + seq.count("C")) / len(seq)) * 100.0

def flexibility_score(seq):
    if len(seq) < 2:
        return 0.0
    pairs = len(seq) - 1
    flexible = 0
    for i in range(pairs):
        dinuc = seq[i : i + 2]
        if dinuc in ("AT", "TA"):
            flexible += 1
    return flexible / pairs if pairs else 0.0

def repeat_density(seq):
    if not seq:
        return 0.0
    repeated = set()
    for m in re.finditer(r"([ACGT])\1{3,}", seq):
        repeated.update(range(m.start(), m.end()))
    for motif_len in (2, 3):
        i = 0
        n = len(seq)
        while i <= n - motif_len * 3:
            motif = seq[i:i+motif_len]
            repeats = 1
            j = i + motif_len
            while j + motif_len <= n and seq[j:j+motif_len] == motif:
                repeats += 1
                j += motif_len
            if repeats >= 3:
                repeated.update(range(i, j))
                i = j
            else:
                i += 1
    return len(repeated) / len(seq)

def melting_temperature(seq):
    a = seq.count("A")
    t = seq.count("T")
    g = seq.count("G")
    c = seq.count("C")
    return 2 * (a + t) + 4 * (g + c)

def analyze_sequence(seq):
    rows = []
    if len(seq) < WINDOW_SIZE:
        return pd.DataFrame(rows)
    for start in range(0, len(seq) - WINDOW_SIZE + 1, STEP_SIZE):
        window = seq[start:start+WINDOW_SIZE]
        gc = gc_content(window)
        at = 100.0 - gc
        flex = flexibility_score(window)
        repeat = repeat_density(window)
        tm = melting_temperature(window)
        fragility = (0.30 * (at / 100.0)) + (0.35 * flex) + (0.35 * repeat) + + ( 0.15 * (1 - (tm / 400)))
        rows.append({
            "Start": start + 1,
            "End": start + len(window),
            "GC Content (%)": round(gc, 2),
            "AT Content (%)": round(at, 2),
            "Flexibility Score": round(flex, 4),
            "Repeat Density": round(repeat, 4),
            "Tm": tm,
            "Fragility Score": round(fragility, 4),
        })
    return pd.DataFrame(rows)

def fragility_band(score):
    if score >= 0.7:
        return "High"
    if score >= 0.45:
        return "Moderate"
    return "Low"


In [ ]:
SAMPLE_FASTA = ">Fragile region demo sample\n" \
    "ATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATAT\n" \
    "GCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGC\n" \
    "CCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCGCCG\n" \
    "ATGCGTATATATGCGTCCGCCGATATATATATCCGCGGCGGATATATATATATATATATATATATAT\n" \
    "ATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATATAT"

# To use your own sequence, replace SAMPLE_FASTA with your sequence string
sequence_text = SAMPLE_FASTA

# If you want to load from a file, uncomment and use the following:
# with open('your_sequence.fasta') as f:
#     sequence_text = f.read()

seq = parse_sequence(sequence_text)
print(f"Sequence length: {len(seq)} bp")

Sequence length: 359 bp


In [ ]:
if len(seq) < WINDOW_SIZE:
    print(f"Sequence must be at least {WINDOW_SIZE} bp long.")
else:
    results = analyze_sequence(seq)
    display(results)
    # Show the window with the highest fragility score
    if not results.empty:
        top_row = results.loc[results['Fragility Score'].idxmax()]
        print("\nMost fragile window:")
        print(top_row)
        print(f"Classification: {fragility_band(top_row['Fragility Score'])}")

,Start,End,GC Content (%),AT Content (%),Flexibility Score,Repeat Density,Tm,Fragility Score
0,1,100,4.0,96.0,0.9596,0.96,208,1.0319
1,21,120,24.0,76.0,0.7576,1.00,248,0.9002
2,41,140,44.0,56.0,0.5556,1.00,288,0.7544
3,61,160,64.0,36.0,0.3535,1.00,328,0.6087
4,81,180,84.0,16.0,0.1515,0.98,368,0.4560
5,101,200,100.0,0.0,0.0000,0.99,400,0.3465
6,121,220,100.0,0.0,0.0000,1.00,400,0.3500
7,141,240,91.0,9.0,0.0707,0.92,382,0.3805
8,161,260,80.0,20.0,0.1616,0.82,360,0.4186
9,181,280,67.0,33.0,0.2828,0.73,334,0.4782



Most fragile window:
Start                  1.0000
End                  100.0000
GC Content (%)         4.0000
AT Content (%)        96.0000
Flexibility Score      0.9596
Repeat Density         0.9600
Tm                   208.0000
Fragility Score        1.0319
Name: 0, dtype: float64
Classification: High
